In [ ]:
import duckdb
import pandas as pd
import numpy as np

# 1. Connect to DuckDB (In-memory)
con = duckdb.connect(database=':memory:')

# 2. Define the path and file mapping for all datasets
# Using a dictionary for better maintainability
data_path = 'data/raw/'
datasets = {
    'app_train': 'application_train.csv',
    'app_test': 'application_test.csv',
    'bureau': 'bureau.csv',
    'bureau_balance': 'bureau_balance.csv',
    'prev_app': 'previous_application.csv',
    'pos_cash': 'POS_CASH_balance.csv',
    'installments': 'installments_payments.csv',
    'credit_card': 'credit_card_balance.csv'
}

# 3. Register all files as DuckDB Views
print("Registering datasets as SQL Views...")
for view_name, file_name in datasets.items():
    full_path = f"{data_path}{file_name}"
    try:
        con.execute(f"CREATE VIEW IF NOT EXISTS {view_name} AS SELECT * FROM read_csv_auto('{full_path}')")
        print(f"  [OK] View '{view_name}' registered.")
    except Exception as e:
        print(f"  [ERROR] Could not register {view_name}: {e}")

# 4. Quick sanity check: count rows for the main tables
print("\n--- Summary ---")
summary_query = """
    SELECT 'Train Set' as table_name, COUNT(*) as row_count FROM app_train
    UNION ALL
    SELECT 'Bureau' as table_name, COUNT(*) as row_count FROM bureau
    UNION ALL
    SELECT 'Prev Applications' as table_name, COUNT(*) as row_count FROM prev_app
"""
display(con.execute(summary_query).df())

Registering datasets as SQL Views...
  [OK] View 'app_train' registered.
  [OK] View 'app_test' registered.
  [OK] View 'bureau' registered.
  [OK] View 'bureau_balance' registered.
  [OK] View 'prev_app' registered.
  [OK] View 'pos_cash' registered.
  [OK] View 'installments' registered.
  [OK] View 'credit_card' registered.

--- Summary ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table_name,row_count
0,Train Set,307511
1,Bureau,1716428
2,Prev Applications,1670214


### bureau.csv

Left Join

In [ ]:
# Aggregating credit history and joining it to the main table
bureau_query = """
    SELECT 
        app.SK_ID_CURR,
        app.TARGET,
        app.AMT_INCOME_TOTAL,
        COUNT(bur.SK_ID_BUREAU) AS bureau_loan_count,
        COALESCE(SUM(bur.AMT_CREDIT_SUM), 0) AS bureau_total_debt,
        COALESCE(SUM(bur.AMT_CREDIT_SUM_OVERDUE), 0) AS bureau_total_overdue
    FROM app_train AS app
    LEFT JOIN bureau AS bur
        ON app.SK_ID_CURR = bur.SK_ID_CURR
    GROUP BY 
        app.SK_ID_CURR, 
        app.TARGET,
        app.AMT_INCOME_TOTAL
    LIMIT 10;
"""

df_features = con.execute(bureau_query).df()
display(df_features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,bureau_loan_count,bureau_total_debt,bureau_total_overdue
0,258494,1,157500.0,11,2893500.000,0.0
1,223857,0,247500.0,4,2693038.500,0.0
2,340468,0,135000.0,3,352098.225,0.0
3,251095,1,112500.0,2,5964711.750,0.0
4,241571,0,270000.0,5,2115405.000,0.0
5,228738,0,135000.0,2,594000.000,0.0
6,292587,0,202500.0,4,232725.240,0.0
7,142520,0,360000.0,5,1638297.000,0.0
8,314602,0,202500.0,13,4641873.840,0.0
9,142924,0,540000.0,6,6424492.500,0.0


In [ ]:
# Creating an Analytical Base Table (ABT)
abt_query = """
SELECT
    app.SK_ID_CURR,
    app.TARGET,
    app.AMT_INCOME_TOTAL,
    COALESCE(bur.total_active_debt, 0) AS total_active_debt,
    COALESCE(bur.total_closed_debt, 0) AS total_closed_debt,
    COALESCE(bur.total_overdue_debt, 0) AS total_overdue_debt,
    COALESCE(bur.total_overdue_debt, 0) / NULLIF(COALESCE(bur.total_active_debt, 0), 0) AS overdue_to_active_ratio
FROM app_train AS app
LEFT JOIN (
    SELECT
        SK_ID_CURR,
        SUM(AMT_CREDIT_SUM) FILTER (WHERE CREDIT_ACTIVE = 'Active') AS total_active_debt,
        SUM(AMT_CREDIT_SUM) FILTER (WHERE CREDIT_ACTIVE = 'Closed') AS total_closed_debt,
        SUM(AMT_CREDIT_SUM_OVERDUE) AS total_overdue_debt
    FROM bureau
    GROUP BY 
        SK_ID_CURR
) AS bur ON app.SK_ID_CURR = bur.SK_ID_CURR;
"""

df_train = con.execute(abt_query).df()
print(f"Dataset uploaded: {df_train.shape[0]} rows and {df_train.shape[1]} columns.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset caricato: 307511 righe e 7 colonne.


In [ ]:
import numpy as np
import pandas as pd

def calculate_woe_iv(df, feature, target):
    # group by the feature and calculate the number of defaults and non-defaults
    lst = []
    for val in df[feature].unique():
        val_df = df[df[feature] == val]
        good = len(val_df[val_df[target] == 0]) # Non-Default
        bad = len(val_df[val_df[target] == 1])  # Default
        lst.append([feature, val, good, bad])
    
    data = pd.DataFrame(lst, columns=['Variable', 'Value', 'Good', 'Bad'])
    
    # Calculate global percentages to avoid divisions by zero
    total_good = data['Good'].sum()
    total_bad = data['Bad'].sum()
    
    # Add a tiny epsilon (0.0001) to avoid log(0) or div/0
    data['Distr_Good'] = (data['Good'] + 0.0001) / total_good
    data['Distr_Bad'] = (data['Bad'] + 0.0001) / total_bad
    
    # Calculate WoE and IV
    data['WoE'] = np.log(data['Distr_Good'] / data['Distr_Bad'])
    data['IV_component'] = (data['Distr_Good'] - data['Distr_Bad']) * data['WoE']
    
    # Sort by WoE to see bins from most risky to safest
    data = data.sort_values(by='WoE')
    
    total_iv = data['IV_component'].sum()
    print(f"Information Value (IV) per '{feature}': {total_iv:.4f}")
    
    return data[['Variable', 'Value', 'Good', 'Bad', 'WoE', 'IV_component']]

In [ ]:
# 1. Boolean masks for binning
conditions = [
    (df_train['total_active_debt'] == 0),
    (df_train['total_active_debt'] > 0) & (df_train['total_active_debt'] <= 100000),
    (df_train['total_active_debt'] > 100000)
]

# 2. Define the labels for each condition (must be in the same order)
choices = ['No Debt', 'Low Debt', 'High Debt']

# 3. Create the new column using np.select
# 'default' serves as a safety net in case a value doesn't fit any condition
df_train['active_debt_bin'] = np.select(conditions, choices, default='Unknown')

# 4. Verify how many clients ended up in each Bin
print("Distribution in Bins:")
print(df_train['active_debt_bin'].value_counts())
print("-" * 30)

# 5. Call the mathematical function to calculate Weight of Evidence (WoE) and Information Value (IV)
woe_iv_results = calculate_woe_iv(df_train, 'active_debt_bin', 'TARGET')

# Visualize the resulting table
display(woe_iv_results)

Distribution in Bins:
active_debt_bin
High Debt    193478
No Debt       93440
Low Debt      20593
Name: count, dtype: int64
------------------------------
Information Value (IV) per 'active_debt_bin': 0.0014


,Variable,Value,Good,Bad,WoE,IV_component
2,active_debt_bin,Low Debt,18747,1846,-0.114473,0.000921
1,active_debt_bin,High Debt,177774,15704,-0.005888,0.000022
0,active_debt_bin,No Debt,86165,7275,0.039335,0.000462


In [ ]:
# 1. Behavioral Binning on overdue debt
conditions_overdue = [
    (df_train['total_overdue_debt'] == 0),
    (df_train['total_overdue_debt'] > 0)
]

choices_overdue = ['Clean History', 'Has Overdue Debt']

df_train['overdue_status'] = np.select(conditions_overdue, choices_overdue, default='Unknown')

# 2. Calculate WoE and IV
woe_iv_overdue = calculate_woe_iv(df_train, 'overdue_status', 'TARGET')
display(woe_iv_overdue)

Information Value (IV) per 'overdue_status': 0.0095


,Variable,Value,Good,Bad,WoE,IV_component
1,overdue_status,Has Overdue Debt,2794,540,-0.788825,0.009362
0,overdue_status,Clean History,279892,24285,0.012059,0.000143
